In [0]:
import pyspark.sql.functions as f
from pyspark.sql import Window

Let's answer some questions about our FruitShop orders data.

Load the data and recall the schema.

In [0]:
df_fruitshop = spark.read.parquet('/Volumes/workspace/default/class/fruitshop.parquet')

df_fruitshop.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- quantity: double (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- items_discount: array (nullable = true)
 |    |-- element: string (containsNull = true)



Answer the following questions. Good luck!

1. What is the general average number of different items sold per order?

In [0]:
(
    df_fruitshop
    .withColumn(
        'unique_item_names',
        f.array_distinct(f.transform(f.col('items'), lambda x: x['name']))
    )
    .withColumn("num_items", f.size("unique_item_names"))
    .select(
        f.avg("num_items").alias("average_items_per_order")
    )
).display()

average_items_per_order
2.5


2. What is the total amount of each fruit sold?

***Hint:*** Start by transforming the data into a more usable format.

In [0]:
(
    df_fruitshop
    .select(f.inline('items'))
    .groupBy('name')
    .agg(f.sum('quantity').alias('total_quantity'))
).display()

name,total_quantity
Apple,5.0
Banana,4.2
Peach,3.0
Cherry,1.5
Strawberry,0.5


3. What was the total amount of `Peach` that was sold in orders where `Peach` was at discount?

In [0]:
(
    df_fruitshop
    # Explode items to get one row per item
    .select(
        'order_id',
        'items_discount',
        f.explode(f.col('items')).alias('item')
    )
    # Filter for Peach items with discount
    .filter(
        (f.col('item.name') == 'Peach')
        & (f.array_contains(f.col('items_discount'), 'Peach'))
    )
    .select(
        f.sum('item.quantity').alias('total_amount')
    )
).display()

total_amount
null


In [0]:
# Check which orders have Peach discounted
df_fruitshop.filter(
    f.array_contains(f.col('items_discount'), 'Peach')
).display()
# Returns 0 rows

order_id,order_date,items,items_discount


4. What is the price of the most expensive item in each order?

In [0]:
(
    df_fruitshop
    # Get item prices in a seperate array
    .withColumn(
        'items_price',
        f.transform(f.col('items'), lambda x: x['price'])
    )
    # Get the maximum value in the prices array
    .withColumn(
        'max_price',
        f.array_max(f.col('items_price'))
    )
).display()

order_id,order_date,items,items_discount,items_price,max_price
8810,2024-05-15,"List(List(Banana, 2.5, 1.99), List(Peach, 1.0, 2.49), List(Apple, 1.0, 2.99), List(Cherry, 0.5, 3.49))",List(Banana),"List(1.99, 2.49, 2.99, 3.49)",3.49
9762,2024-05-02,"List(List(Strawberry, 0.5, 6.99), List(Apple, 3.0, 2.99), List(Cherry, 1.0, 3.49))","List(Apple, Cherry)","List(6.99, 2.99, 3.49)",6.99
5642,2024-05-18,"List(List(Apple, 1.0, 2.99), List(Banana, 1.7, 1.99))",List(Apple),"List(2.99, 1.99)",2.99
1234,2024-05-10,"List(List(Peach, 2.0, 2.49))",List(),List(2.49),2.49


5. What is the name of the most expensive item in each order?

***Hint:*** Use a window function.

In [0]:
window = Window.partitionBy('order_id')

(
    df_fruitshop
    .select(
        'order_id',
        f.inline('items')
    )
    .withColumn(
        'max_price',
        f.max(f.col('price')).over(window)
    )
    .filter(
        f.col('price') == f.col('max_price')
    )
    .dropDuplicates()
    .select(
        'order_id',
        'name'
    )
).display()

order_id,name
1234,Peach
5642,Apple
8810,Cherry
9762,Strawberry


6. What is the total amount of each fruit that is sold in orders where the number of unique items is greater than the general average?

Let's break down the question:
- Calculate the general average number of unique items sold per order.
- Filter the orders where the number of unique items is greater than the general average.
- Calculate the total amount of each fruit sold in the filtered orders.

In [0]:
# Get the average number of items per order

avg_items_per_order = (
    df_fruitshop
    .withColumn(
        'unique_item_names',
        f.array_distinct(f.transform(f.col('items'), lambda x: x['name']))
    )
    .withColumn("unique_num_items", f.size("unique_item_names"))
    .select(
        f.avg("unique_num_items").alias("average_items_per_order")
    )
).collect()[0][0]


(
    df_fruitshop
    .withColumn('avg_items_per_order', f.lit(avg_items_per_order))
    .filter(
        f.size(
            f.array_distinct(
                f.transform(f.col('items'), lambda x: x['name'])
            )
        ) > f.col('avg_items_per_order')
    )
    .select(
        'order_id',
        f.explode('items').alias('item')
    )
    .groupBy('item.name')
    .agg(f.sum('item.quantity').alias('total_quantity'))

).display()

name,total_quantity
Apple,4.0
Banana,2.5
Peach,1.0
Cherry,1.5
Strawberry,0.5
